Тут я попробую написать свою реализацию беггинга на дереьях и бустинга  


In [6]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.datasets import fetch_california_housing
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV
import numpy as np
import pandas as pd


In [7]:
class RNForest(RegressorMixin, BaseEstimator):
    def __init__(self, max_trees=300, max_depth=3):
        self.max_trees = max_trees
        self.max_depth = max_depth
        self.models = []

    def fit(self, X, y):
        self.models = []
        n_samples = X.shape[0]
        for i in range(self.max_trees):
            ind = np.random.choice(n_samples, size=n_samples, replace=True)
            X_new , y_new = X[ind], y[ind]
            tree = DecisionTreeRegressor(max_depth=self.max_depth)
            tree.fit(X_new, y_new)
            self.models.append(tree)
        return self

    def predict(self, X):
        predictions = []
        for tree in self.models:
            predictions.append(tree.predict(X))
        return np.mean(predictions, axis=0)

Смотреть на результаты буду на сете california houses

In [8]:
raw_data = fetch_california_housing()
df = pd.DataFrame(raw_data.data, columns=raw_data.feature_names)
df['MedHouseVal'] = raw_data.target
print(df.head(3))
df.describe()

   MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0  8.3252      41.0  6.984127   1.023810       322.0  2.555556     37.88   
1  8.3014      21.0  6.238137   0.971880      2401.0  2.109842     37.86   
2  7.2574      52.0  8.288136   1.073446       496.0  2.802260     37.85   

   Longitude  MedHouseVal  
0    -122.23        4.526  
1    -122.22        3.585  
2    -122.24        3.521  


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
count,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000
mean,3.870671,28.639486,5.429000,1.096675,1425.476744,3.070655,35.631861,-119.569704,2.068558
std,1.899822,12.585558,2.474173,0.473911,1132.462122,10.386050,2.135952,2.003532,1.153956
min,0.499900,1.000000,0.846154,0.333333,3.000000,0.692308,32.540000,-124.350000,0.149990
25%,2.563400,18.000000,4.440716,1.006079,787.000000,2.429741,33.930000,-121.800000,1.196000
50%,3.534800,29.000000,5.229129,1.048780,1166.000000,2.818116,34.260000,-118.490000,1.797000
75%,4.743250,37.000000,6.052381,1.099526,1725.000000,3.282261,37.710000,-118.010000,2.647250
max,15.000100,52.000000,141.909091,34.066667,35682.000000,1243.333333,41.950000,-114.310000,5.000010


In [9]:
X = df.drop(columns=['MedHouseVal'])
y = df['MedHouseVal']
X_train, X_test, y_train, y_test = train_test_split(X.values, y.values, random_state=42)

myforest = RNForest(max_trees=500, max_depth=4)
myforest.fit(X_train, y_train)
predictions = myforest.predict(X_test)

print(f"MAE: {mean_absolute_error(y_test, predictions)}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, predictions))}")
print(f"r2-score: {r2_score(y_test, predictions)}")

MAE: 0.533483938779714
RMSE: 0.7233223879010476
r2-score: 0.6046039009772868


In [ ]:
params = {
    'max_trees': [200, 300, 400],
    'max_depth': [3,4,5,6,7]
}

grid = GridSearchCV(RNForest(), param_grid=params, scoring='r2', cv=3, n_jobs=-1)
grid.fit(X_train, y_train)

print(f"Лучшие параметры: {grid.best_params_}")
print(f"Лучший RMSE на кросс-валидации: {grid.best_score_:.4f}")

Лучшие параметры: {'max_depth': 7, 'max_trees': 400}
Лучший RMSE на кросс-валидации: -0.7283
